In [1]:
# Parses the rollout data from demo.py and saves them as numpy arrays of states, actions, and pot handle positions (the conditional vector data)

import numpy as np
import csv
import matplotlib.pyplot as plt
import pickle as pkl
import torch
from scipy.spatial.transform import Rotation as R
from transform_utils import quat_to_rot6d, rotvec_to_rot6d, rot6d_to_quat
from diffusers import AutoencoderKL
import os
from net import TimMResNet18Encoder
from net import StateEncoder


/home/icon-labtop/anaconda3/envs/crazy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# Example usage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = TimMResNet18Encoder(pretrained=True, latent_dim=128).to(device)
state_encoder = StateEncoder(input_dim = 7, latent_dim = 32).to(device)
encoder.eval()

TimMResNet18Encoder(
  (backbone): FeatureListNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act1): ReLU(inplace=True)
        (aa): Identity()
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=

In [ ]:
expert_states_list = []
expert_actions_list = []
pot_start_list = []
pot_states_list1 = []
pot_states_list2 = []
imagestate_latents_list0 = []
imagestate_latents_list1 = []

for i in [2, 3]:
    for j in range(1):
        with open("rollouts/newslower/rollout_seed%s_mode%s.pkl" % (j*10, i), "rb") as f:
            rollout = pkl.load(f)
            obs = rollout["observations"]
            actions = np.array(rollout["actions"])
            print("iteration" + str(j))
            pot1 = np.array(rollout["pot_states1"])
            pot2 = np.array(rollout["pot_states2"])
            pot = np.array(rollout["pot_start"])

            pot_start_list.append(np.concatenate((pot[0], pot[1])))

            T_target = 700

            if "camera_obs0" in rollout and "camera_obs1" and "observations" in rollout:
                camera0_obs = np.array(rollout["camera_obs0"])  # (T, H, W, C)
                camera1_obs = np.array(rollout["camera_obs1"])
                observations = rollout["observations"] # (649, Dictionary ())
                T = len(observations)

                if camera0_obs.shape[0] > T_target:
                    camera0_obs = camera0_obs[:T_target]
                    camera1_obs = camera1_obs[:T_target]
                elif camera0_obs.shape[0] < T_target:
                    repeats_needed = T_target - camera0_obs.shape[0]
                    camera0_obs = np.vstack([camera0_obs, np.repeat(camera0_obs[-1][None], repeats_needed, axis=0)])
                    camera1_obs = np.vstack([camera1_obs, np.repeat(camera1_obs[-1][None], repeats_needed, axis=0)])

                if T > T_target:
                    observations = observations[:T_target]
                elif T < T_target:
                    repeats_needed = T_target - T
                    last_obs = observations[-1]
                    for _ in range(repeats_needed):
                        observations.append(last_obs.copy())
                camera0_latents = []
                camera1_latents = []

                for frame_idx in range(T_target):
                    img0 = torch.from_numpy(camera0_obs[frame_idx]).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
                    img1 = torch.from_numpy(camera1_obs[frame_idx]).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
                    
                    robot0_eef_pos = np.array(obs[frame_idx]["robot0_eef_pos"])
                    robot0_eef_quat = np.array(obs[frame_idx]["robot0_eef_quat_site"])
                    robot0_eef_rotvec = R.from_quat(robot0_eef_quat).as_rotvec()
                    robot0_gripper_pos = np.array(obs[frame_idx]["robot0_gripper_pos"]).reshape(-1)

                    robot1_eef_pos = np.array(obs[frame_idx]["robot1_eef_pos"])
                    robot1_eef_quat = np.array(obs[frame_idx]["robot1_eef_quat_site"])
                    robot1_eef_rotvec = R.from_quat(robot1_eef_quat).as_rotvec()
                    robot1_gripper_pos = np.array(obs[frame_idx]["robot1_gripper_pos"]).reshape(-1)


                    state_vec0 = np.hstack([robot0_eef_pos, robot0_eef_rotvec, robot0_gripper_pos])
                    state_vec1 = np.hstack([robot1_eef_pos, robot1_eef_rotvec, robot1_gripper_pos])
                    
                    state_vec_tensor0 = torch.FloatTensor(state_vec0).unsqueeze(0).to(device)
                    state_vec_tensor1 = torch.FloatTensor(state_vec1).unsqueeze(0).to(device)

                    with torch.no_grad():
                        latents0 = encoder(img0)
                        latents1 = encoder(img1)
                        latent_state0 = state_encoder(state_vec_tensor0)
                        latent_state1 = state_encoder(state_vec_tensor1)

                    
                    latent_state0 = latent_state0.cpu().numpy().squeeze() 
                    latent_state1 = latent_state1.cpu().numpy().squeeze()
                    latents0 = latents0.cpu().numpy().squeeze()
                    latents1 = latents1.cpu().numpy().squeeze()

                    latent_concat0 = np.concatenate([latents0, latent_state0])
                    latent_concat1 = np.concatenate([latents1, latent_state1])

                    camera0_latents.append(latent_concat0)
                    camera1_latents.append(latent_concat1)


                camera0_latents = np.array(camera0_latents)
                camera1_latents = np.array(camera1_latents)
            else:
                print("Womp womp :(")

            
            robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
            robot0_eef_quat = np.array([o["robot0_eef_quat_site"] for o in obs])
            robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
            robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
            robot1_eef_quat = np.array([o["robot1_eef_quat_site"] for o in obs])
            robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

            repeats_needed = 700 - actions.shape[0]

            repeated_last = np.tile(actions[-1], (repeats_needed, 1))
            actions = np.vstack([actions, repeated_last])

            repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
            robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
            state = robot0_eef_pos

            repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
            robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
            robot0_eef_rotvec = R.from_quat(robot0_eef_quat).as_rotvec()
            state = np.hstack([state, robot0_eef_rotvec])


            repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
            robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
            robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
            state = np.hstack([state, robot0_gripper_pos])

            repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
            robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
            state = np.hstack([state, robot1_eef_pos])

            repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
            robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
            robot1_eef_rotvec = R.from_quat(robot1_eef_quat).as_rotvec()
            state = np.hstack([state, robot1_eef_rotvec])

            repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
            robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
            robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
            state = np.hstack([state, robot1_gripper_pos])

            repeated_last = np.tile(pot1[-1], (repeats_needed, 1))
            pot1 = np.vstack([pot1, repeated_last])
            repeated_last = np.tile(pot2[-1], (repeats_needed, 1))
            pot2 = np.vstack([pot2, repeated_last])
            pot_states_list1.append(pot1)
            pot_states_list2.append(pot2)

            # --- ENFORCE FINAL SHAPE FOR LATENTS --- #
            T_target = 700

            # Camera 0
            if camera0_latents.shape[0] > T_target:
                camera0_latents = camera0_latents[:T_target]
            elif camera0_latents.shape[0] < T_target:
                repeats_needed = T_target - camera0_latents.shape[0]
                last_frame = camera0_latents[-1:]
                padding = np.repeat(last_frame, repeats_needed, axis=0)
                camera0_latents = np.concatenate([camera0_latents, padding], axis=0)

            if camera1_latents.shape[0] > T_target:
                camera1_latents = camera1_latents[:T_target]
            elif camera1_latents.shape[0] < T_target:
                repeats_needed = T_target - camera1_latents.shape[0]
                last_frame = camera1_latents[-1:]
                padding = np.repeat(last_frame, repeats_needed, axis=0)
                camera1_latents = np.concatenate([camera1_latents, padding], axis=0)

            print("ight blud")
            print(camera0_latents.shape)
            print(state.shape)
                

            imagestate_latents_list0.append(camera0_latents)
            imagestate_latents_list1.append(camera1_latents)
            expert_states_list.append(state)
            expert_actions_list.append(actions)
            


expert_states_rotvec = np.stack(expert_states_list, axis=0)
expert_actions_rotvec = np.stack(expert_actions_list, axis=0)
pot_states_rotvec1 = np.stack(pot_states_list1, axis=0)
pot_states_rotvec2 = np.stack(pot_states_list2, axis=0)
pot_start_rotvec = np.stack(pot_start_list, axis=0)
imagestate_latents_rotvec0 = np.stack(imagestate_latents_list0, axis=0)
imagestate_latents_rotvec1 = np.stack(imagestate_latents_list1, axis=0)

iteration0

ight blud
(700, 160)
(751, 14)
iteration0

ight blud
(700, 160)
(815, 14)


ValueError: all input arrays must have the same shape

In [ ]:
print(np.shape(expert_states_rotvec))
print(np.shape(expert_actions_rotvec))
print(np.shape(pot_states_rotvec1))
print(np.shape(pot_states_rotvec2))
print(np.shape(pot_start_rotvec))
print(np.shape(imagestate_latents_rotvec0))
print(np.shape(imagestate_latents_rotvec1))

In [ ]:
import os
os.makedirs("dataLatentImageStates", exist_ok=True)
np.save("data/models/VAE_models_ICON/TrainingDataDiffusion2/expert_states_newslower_20.npy", expert_states_rotvec)
np.save("data/models/VAE_models_ICON/TrainingDataDiffusion2/expert_actions_newslower_20.npy", expert_actions_rotvec)
np.save("data/models/VAE_models_ICON/TrainingDataDiffusion2/pot_states1_newslower_20.npy", pot_states_rotvec1)
np.save("data/models/VAE_models_ICON/TrainingDataDiffusion2/pot_states2_newslower_20.npy", pot_states_rotvec2)
np.save("data/models/VAE_models_ICON/TrainingDataDiffusion2/pot_start_newslower_20.npy", pot_start_rotvec)
np.save("data/models/VAE_models_ICON/TrainingDataDiffusion2/arm1_images_latents.npy", imagestate_latents_rotvec0)
np.save("data/models/VAE_models_ICON/TrainingDataDiffusion2/arm2_images_latents.npy", imagestate_latents_rotvec1)

In [ ]:
expert_states_list = []
expert_actions_list = []
pot_start_list = []
pot_states_list1 = []
pot_states_list2 = []
image_state_latents_list = []

for i in [2, 3]:
    for j in range(100):
        with open(f"rollouts/newslower/rollout_seed{j*10}_mode{i}.pkl", "rb") as f:
            rollout = pkl.load(f)

        obs = rollout["observations"]           # List of dicts
        actions = np.array(rollout["actions"])
        print(f"Iteration {j}, actions shape: {actions.shape}")

        pot1 = np.array(rollout["pot_states1"])
        pot2 = np.array(rollout["pot_states2"])
        pot = np.array(rollout["pot_start"])
        pot_start_list.append(np.concatenate((pot[0], pot[1])))

        T_target = 700

        # Process camera images
        if "camera_obs0" in rollout and "camera_obs1" in rollout:
            camera0_obs = np.array(rollout["camera_obs0"])  # (T, H, W, C)
            camera1_obs = np.array(rollout["camera_obs1"])
            T = len(obs)

            # Pad or truncate images
            if camera0_obs.shape[0] > T_target:
                camera0_obs = camera0_obs[:T_target]
                camera1_obs = camera1_obs[:T_target]
            elif camera0_obs.shape[0] < T_target:
                repeats_needed = T_target - camera0_obs.shape[0]
                camera0_obs = np.vstack([camera0_obs, np.repeat(camera0_obs[-1][None], repeats_needed, axis=0)])
                camera1_obs = np.vstack([camera1_obs, np.repeat(camera1_obs[-1][None], repeats_needed, axis=0)])

            # Pad observations
            if T > T_target:
                obs = obs[:T_target]
            elif T < T_target:
                last_obs = obs[-1].copy()
                obs += [last_obs] * (T_target - T)

            states_and_latents = []

            # Encode each frame and concat with robot states
            for frame_idx in range(T_target):
                img0 = torch.from_numpy(camera0_obs[frame_idx]).permute(2,0,1).unsqueeze(0).float().to(device) / 255.0
                img1 = torch.from_numpy(camera1_obs[frame_idx]).permute(2,0,1).unsqueeze(0).float().to(device) / 255.0

                # Extract states
                robot0_eef_pos = np.array(obs[frame_idx]["robot0_eef_pos"])
                robot0_eef_quat = np.array(obs[frame_idx]["robot0_eef_quat_site"])
                robot0_eef_rotvec = R.from_quat(robot0_eef_quat).as_rotvec()
                robot0_gripper_pos = np.array(obs[frame_idx]["robot0_gripper_pos"]).reshape(-1)

                robot1_eef_pos = np.array(obs[frame_idx]["robot1_eef_pos"])
                robot1_eef_quat = np.array(obs[frame_idx]["robot1_eef_quat_site"])
                robot1_eef_rotvec = R.from_quat(robot1_eef_quat).as_rotvec()
                robot1_gripper_pos = np.array(obs[frame_idx]["robot1_gripper_pos"]).reshape(-1)

                state_vec = np.hstack([robot0_eef_pos, robot0_eef_rotvec, robot0_gripper_pos,
                                       robot1_eef_pos, robot1_eef_rotvec, robot1_gripper_pos])

                with torch.no_grad():
                    latents0 = encoder(img0).cpu().numpy().squeeze()
                    latents1 = encoder(img1).cpu().numpy().squeeze()

                # Concatenate state + both camera latents
                full_vec = np.hstack([state_vec, latents0, latents1])
                states_and_latents.append(full_vec)

            states_and_latents = np.stack(states_and_latents, axis=0)

        else:
            print("No camera observations found, skipping rollout.")
            continue

        # Pad actions
        if actions.shape[0] < T_target:
            repeats_needed = T_target - actions.shape[0]
            actions = np.vstack([actions, np.tile(actions[-1], (repeats_needed, 1))])

        # Pad pot states
        for arr, lst in zip([pot1, pot2], [pot_states_list1, pot_states_list2]):
            if arr.shape[0] < T_target:
                arr = np.vstack([arr, np.tile(arr[-1], (T_target - arr.shape[0], 1))])
            lst.append(arr)

        expert_states_list.append(states_and_latents)
        expert_actions_list.append(actions)

# Stack everything
expert_states_rotvec = np.stack(expert_states_list, axis=0)
expert_actions_rotvec = np.stack(expert_actions_list, axis=0)
pot_states_rotvec1 = np.stack(pot_states_list1, axis=0)
pot_states_rotvec2 = np.stack(pot_states_list2, axis=0)
pot_start_rotvec = np.stack(pot_start_list, axis=0)

print("Final shapes:")
print(expert_states_rotvec.shape, expert_actions_rotvec.shape)
